# SyLOC-T : Projet BDA (CROUS de Thiès)
## Notebook d'analyse BDA et deploiement cloud

Ce notebook permet d'initialiser l'environnement, executer les requetes SQL BDA et lancer le service.

### 1. Clonage du depot et installation des dependances

In [ ]:
import os
from getpass import getpass

# Si le depot est prive, entrer le token GitHub
token = getpass("GitHub Token (laisser vide si public) : ").strip()

!rm -rf SyLOC-T
if token:
    !git clone -b develop https://{token}@github.com/mhdlamine21/SyLOC-T.git
else:
    !git clone -b develop https://github.com/mhdlamine21/SyLOC-T.git

%cd SyLOC-T

!pip install -r vcn_backend/requirements.txt
!pip install pandas matplotlib seaborn pyngrok

### 2. Initialisation de la base de donnees (SQLite)

In [ ]:
import os

os.environ["DB_ENGINE"] = "sqlite"
with open("vcn_backend/.env", "w") as f:
    f.write("SECRET_KEY=django-insecure-colab-key-syloc\n")
    f.write("DEBUG=True\n")
    f.write("DB_ENGINE=sqlite\n")
    f.write("ALLOWED_HOSTS=*\n")
    f.write("CORS_ALLOWED_ORIGINS=http://localhost:5173,http://localhost:3000\n")

%cd vcn_backend
!python manage.py migrate
!python seed.py
%cd ..

### 3. Requetes SQL BDA et visualisations

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('vcn_backend/db.sqlite3')

# Requete 1 : Repartition du patrimoine et disponibilite
query_locaux = """
SELECT 
    type_local AS Type,
    CASE WHEN est_libre = 1 THEN 'Disponible' ELSE 'Occupe' END AS Disponibilite,
    etat_physique AS Etat,
    COUNT(*) AS Total,
    ROUND(AVG(surface_m2), 1) AS Surface_Moy_m2
FROM patrimoine_local
GROUP BY type_local, est_libre, etat_physique
ORDER BY Total DESC;
"""
df_locaux = pd.read_sql_query(query_locaux, conn)
print("1. Analyse du parc immobilier")
display(df_locaux)

plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")
sns.barplot(data=df_locaux, x='Type', y='Total', hue='Disponibilite', palette='Set2')
plt.title('Repartition des locaux par type et disponibilite')
plt.ylabel('Nombre de locaux')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# Requete 2 : Repartition des reglements par mode de paiement
query_paiements = """
SELECT 
    mode AS Mode_Paiement,
    statut AS Statut_Paiement,
    COUNT(*) AS Nombre_Transactions,
    ROUND(SUM(montant_regle), 0) AS Total_Encaisse_FCFA
FROM paiements_paiement
GROUP BY mode, statut;
"""
df_paiements = pd.read_sql_query(query_paiements, conn)
print("\n2. Analyse des reglements")
display(df_paiements)

if not df_paiements.empty and df_paiements['Total_Encaisse_FCFA'].sum() > 0:
    plt.figure(figsize=(6, 6))
    plt.pie(df_paiements['Total_Encaisse_FCFA'], labels=df_paiements['Mode_Paiement'], autopct='%1.1f%%', colors=['#3498db','#e74c3c','#2ecc71'])
    plt.title('Repartition des encaissements par mode')
    plt.show()

### 4. Deploiement cloud avec Ngrok

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from getpass import getpass

ngrok_token = getpass("Authtoken Ngrok : ").strip()
ngrok.set_auth_token(ngrok_token)

print("Lancement du serveur Django...")
backend_proc = subprocess.Popen(["python", "vcn_backend/manage.py", "runserver", "0.0.0.0:8000"])
time.sleep(3)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print(f"Documentation Swagger : {public_url}/api/docs/")
print(f"Administration Django : {public_url}/admin/")
print(f"Statistiques BDA      : {public_url}/api/public/stats/")
print(f"Vitrine publique      : {public_url}/api/public/vitrine/")